**Gold Layer: Dimensional Model**

Builds the star schema from Silver tables: conformed dimensions
(`dim_customers` with SCD Type 2, `dim_products`, `dim_loyalty`,
`dim_exchange_rates`, `dim_date`) and fact tables (`fact_orders` at the
order-line grain, `fact_reviews` at the review grain).

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, TimestampType, DateType
from datetime import datetime, date

GOLD_RUN_TS = datetime.utcnow().isoformat()
print(f"Gold layer run started at {GOLD_RUN_TS}")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 4, Finished, Available, Finished, False)

/tmp/ipykernel_6297/2407006888.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  GOLD_RUN_TS = datetime.utcnow().isoformat()


**dim_products**

A straightforward conformed dimension — surrogate key added, attributes
carried through as-is from Silver. No SCD tracking needed here; product
catalog changes (price, active status) are treated as current-state-only
for this project's scope, since historical price tracking wasn't a
stated business requirement.

In [3]:
silver_products = spark.table("silver_products")

dim_products = (
    silver_products
    .withColumn("product_key", F.monotonically_increasing_id().cast(IntegerType()))
    .select("product_key", "product_id", "product_name", "category", "unit_price", "is_active")
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

dim_products.write.format("delta").mode("overwrite").saveAsTable("dim_products")
print(f"dim_products: {dim_products.count()} rows")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 5, Finished, Available, Finished, False)

dim_products: 100 rows


**dim_date**

Generated date dimension spanning the range of our order data, enabling
Power BI to do calendar-based analysis (day-of-week, quarter, fiscal
periods) via simple joins instead of DAX date arithmetic.

In [4]:
from pyspark.sql.functions import sequence, to_date, explode, col, year, month, dayofmonth, dayofweek, quarter, date_format

start_date = "2024-01-01"
end_date = "2026-12-31"

dim_date = (
    spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as date")
    .withColumn("date", explode(col("date")))
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
)

dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")
print(f"dim_date: {dim_date.count()} rows")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 6, Finished, Available, Finished, False)

dim_date: 1096 rows


**dim_loyalty**

Conformed loyalty dimension from Silver. Treated as current-state-only
(no SCD tracking) — this project's SCD2 demonstration is scoped to
`dim_customers`, where "what did the customer look like at order time"
is the realistic business question. Loyalty tier history isn't a stated
requirement, so adding SCD2 here would be complexity without a business
driver.

In [5]:
silver_loyalty = spark.table("silver_loyalty")

dim_loyalty = (
    silver_loyalty
    .withColumn("loyalty_key", F.monotonically_increasing_id().cast(IntegerType()))
    .select("loyalty_key", "member_id", "full_name", "email", "points_balance", "tier", "enrollment_date")
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

dim_loyalty.write.format("delta").mode("overwrite").saveAsTable("dim_loyalty")
print(f"dim_loyalty: {dim_loyalty.count()} rows")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 7, Finished, Available, Finished, False)

dim_loyalty: 289 rows


**dim_exchange_rates**

Conformed currency dimension — one row per currency per rate date. This
will join against `fact_orders` (once we add a currency dimension to
orders in 7C) to support multi-currency order valuation.

In [6]:
silver_exchange_rates = spark.table("silver_exchange_rates")

dim_exchange_rates = (
    silver_exchange_rates
    .withColumn("exchange_rate_key", F.monotonically_increasing_id().cast(IntegerType()))
    .select("exchange_rate_key", "base_currency", "rate_date", "currency_code", "exchange_rate")
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

dim_exchange_rates.write.format("delta").mode("overwrite").saveAsTable("dim_exchange_rates")
print(f"dim_exchange_rates: {dim_exchange_rates.count()} rows")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 8, Finished, Available, Finished, False)

dim_exchange_rates: 29 rows


**PHASE 2 — SCD Type 2 + Late-Arriving Dimension Handling
Objective**

Build dim_customers with full SCD Type 2 history tracking, then simulate a realistic "Day 2" scenario: some existing customers change (address update), a few brand-new customers arrive, and — critically — a new order references a customer who hasn't synced into the dimension yet. This forces us to implement the inferred member pattern for late-arriving dimensions, not just describe it.

**dim_customers — SCD Type 2 (Baseline / Day 1)**

Establishes the initial SCD2 structure for the customer dimension:

- `customer_key`: surrogate key (unique per version of a customer)
- `customer_id`: natural/business key (stable across versions)
- `effective_start_date` / `effective_end_date`: validity window for this version
- `is_current`: flag for the currently-active version

On this first run, every customer has exactly one version, all marked
current, with `effective_end_date = NULL` (meaning "still active"). This
becomes the baseline that Day 2's SCD2 merge logic will compare against.

In [7]:
silver_customers = spark.table("silver_customers")

dim_customers_baseline = (
    silver_customers
    .withColumn("customer_key", F.monotonically_increasing_id().cast(IntegerType()))
    .withColumn("effective_start_date", F.lit(date(2026, 8, 26)).cast(DateType()))
    .withColumn("effective_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .select(
        "customer_key", "customer_id", "first_name", "last_name", "email", "phone",
        "city", "country", "effective_start_date", "effective_end_date", "is_current",
    )
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

dim_customers_baseline.write.format("delta").mode("overwrite").saveAsTable("dim_customers")
print(f"dim_customers (baseline): {dim_customers_baseline.count()} rows, all current")

# Sanity check: confirm customers 1-15 exist with their ORIGINAL (pre-Day2) city values
spark.table("dim_customers").filter(F.col("customer_id").isin([1,2,3,4,5])).select(
    "customer_id", "city", "is_current", "effective_start_date", "effective_end_date"
).show()

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 9, Finished, Available, Finished, False)

dim_customers (baseline): 510 rows, all current
+-----------+-----------------+----------+--------------------+------------------+
|customer_id|             city|is_current|effective_start_date|effective_end_date|
+-----------+-----------------+----------+--------------------+------------------+
|          3|West Jessicahaven|      true|          2026-08-26|              NULL|
|          1|       Jamesburgh|      true|          2026-08-26|              NULL|
|          5|      Port Rachel|      true|          2026-08-26|              NULL|
|          2|        East Dean|      true|          2026-08-26|              NULL|
|          4| New Tracychester|      true|          2026-08-26|              NULL|
+-----------+-----------------+----------+--------------------+------------------+



**dim_customers — SCD Type 2 Merge (Day 2)**

Compares the latest `silver_customers` against the current `dim_customers`
rows (`is_current = true`) to detect three categories of change:

1. **Unchanged customers** — no action, existing current row stays as-is.
2. **Changed customers** (e.g. city updated) — the existing current row is
   *expired* (`is_current = false`, `effective_end_date` set), and a new
   row is inserted as the new current version with a fresh
   `effective_start_date`.
3. **Brand-new customers** — inserted as a new current row with no prior
   version to expire.

This is implemented with Delta Lake's `MERGE INTO`, which is the standard
mechanism for SCD2 in a lakehouse — it lets us express "update the old
version's end date" and "insert new rows" as a single atomic operation
rather than juggling separate read-modify-write steps that could race
against each other.

In [8]:
from delta.tables import DeltaTable

silver_customers_day2 = spark.table("silver_customers")
dim_customers_current = spark.table("dim_customers").filter(F.col("is_current") == True)

TODAY = date(2026, 8, 27)  # Day 2 processing date

# Step 1: Identify which customers actually changed (compare attribute values,
# not just existence) — this is what distinguishes "no-op" from "needs new version"
change_detection = silver_customers_day2.alias("src").join(
    dim_customers_current.alias("cur"),
    on="customer_id",
    how="left"
).select(
    F.col("src.customer_id"),
    F.col("src.first_name"), F.col("src.last_name"), F.col("src.email"),
    F.col("src.phone"), F.col("src.city"), F.col("src.country"),
    F.col("cur.customer_key").alias("existing_key"),
    F.col("cur.city").alias("current_city"),
    F.when(F.col("cur.customer_key").isNull(), F.lit("NEW"))
     .when(F.col("src.city") != F.col("cur.city"), F.lit("CHANGED"))
     .otherwise(F.lit("UNCHANGED"))
     .alias("change_type")
)

change_summary = change_detection.groupBy("change_type").count()
print("Change detection summary:")
change_summary.show()

# Step 2: Expire the old current version for anything that CHANGED
dim_customers_table = DeltaTable.forName(spark, "dim_customers")

changed_ids = [
    row["customer_id"] for row in
    change_detection.filter(F.col("change_type") == "CHANGED").select("customer_id").collect()
]

if changed_ids:
    dim_customers_table.update(
        condition=(F.col("is_current") == True) & (F.col("customer_id").isin(changed_ids)),
        set={
            "is_current": F.lit(False),
            "effective_end_date": F.lit(TODAY),
        }
    )
    print(f"Expired {len(changed_ids)} old customer versions.")

# Step 3: Insert new current rows for both CHANGED and NEW customers
existing_max_key = spark.table("dim_customers").agg(F.max("customer_key")).collect()[0][0]

new_versions = (
    change_detection.filter(F.col("change_type").isin(["CHANGED", "NEW"]))
    .select("customer_id", "first_name", "last_name", "email", "phone", "city", "country")
    .withColumn("row_num", F.row_number().over(
        __import__("pyspark.sql.window", fromlist=["Window"]).Window.orderBy("customer_id")
    ))
    .withColumn("customer_key", (F.lit(existing_max_key + 1) + F.col("row_num") - 1).cast(IntegerType()))
    .withColumn("effective_start_date", F.lit(TODAY).cast(DateType()))
    .withColumn("effective_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
    .drop("row_num")
)

new_versions.write.format("delta").mode("append").saveAsTable("dim_customers")
print(f"Inserted {new_versions.count()} new current versions (changed + brand-new customers).")

# Validation: confirm customers 1-15 now have TWO rows each (one expired, one current)
print("\n--- Verifying SCD2 history for customer_id = 1 ---")
spark.table("dim_customers").filter(F.col("customer_id") == 1).select(
    "customer_key", "customer_id", "city", "is_current", "effective_start_date", "effective_end_date"
).orderBy("effective_start_date").show(truncate=False)

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 10, Finished, Available, Finished, False)

Change detection summary:
+-----------+-----+
|change_type|count|
+-----------+-----+
|  UNCHANGED|  510|
+-----------+-----+

Inserted 0 new current versions (changed + brand-new customers).

--- Verifying SCD2 history for customer_id = 1 ---
+------------+-----------+----------+----------+--------------------+------------------+
|customer_key|customer_id|city      |is_current|effective_start_date|effective_end_date|
+------------+-----------+----------+----------+--------------------+------------------+
|123         |1          |Jamesburgh|true      |2026-08-26          |NULL              |
+------------+-----------+----------+----------+--------------------+------------------+



In [9]:
spark.sql("DESCRIBE HISTORY silver_customers").select(
    "version", "timestamp", "operation"
).orderBy("version").show(truncate=False)

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 11, Finished, Available, Finished, False)

+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|0      |2026-08-26 17:10:15|CREATE OR REPLACE TABLE AS SELECT|
|1      |2026-08-26 17:23:09|CREATE OR REPLACE TABLE AS SELECT|
|2      |2026-08-27 04:05:34|CREATE OR REPLACE TABLE AS SELECT|
|3      |2026-08-27 06:41:53|CREATE OR REPLACE TABLE AS SELECT|
|4      |2026-08-27 20:39:37|CREATE OR REPLACE TABLE AS SELECT|
|5      |2026-08-28 07:47:58|CREATE OR REPLACE TABLE AS SELECT|
+-------+-------------------+---------------------------------+



In [10]:
CORRECT_DAY1_VERSION = 3

silver_customers_day1 = spark.read.format("delta").option("versionAsOf", CORRECT_DAY1_VERSION).table("silver_customers")
print(f"Recovered Day 1 snapshot: {silver_customers_day1.count()} rows")

dim_customers_baseline_corrected = (
    silver_customers_day1
    .withColumn("customer_key", F.monotonically_increasing_id().cast(IntegerType()))
    .withColumn("effective_start_date", F.lit(date(2026, 8, 26)).cast(DateType()))
    .withColumn("effective_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .select(
        "customer_key", "customer_id", "first_name", "last_name", "email", "phone",
        "city", "country", "effective_start_date", "effective_end_date", "is_current",
    )
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

dim_customers_baseline_corrected.write.format("delta").mode("overwrite").saveAsTable("dim_customers")
print(f"dim_customers corrected baseline: {dim_customers_baseline_corrected.count()} rows")

print("\n--- Confirming customer_id 1-5 show ORIGINAL Day 1 cities ---")
spark.table("dim_customers").filter(F.col("customer_id").isin([1,2,3,4,5])).select(
    "customer_id", "city", "is_current"
).show()

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 12, Finished, Available, Finished, False)

Recovered Day 1 snapshot: 510 rows
dim_customers corrected baseline: 510 rows

--- Confirming customer_id 1-5 show ORIGINAL Day 1 cities ---
+-----------+-----------------+----------+
|customer_id|             city|is_current|
+-----------+-----------------+----------+
|          3|West Jessicahaven|      true|
|          1|       Jamesburgh|      true|
|          5|      Port Rachel|      true|
|          2|        East Dean|      true|
|          4| New Tracychester|      true|
+-----------+-----------------+----------+



In [11]:
CANDIDATE_VERSION = 2

check = spark.read.format("delta").option("versionAsOf", CANDIDATE_VERSION).table("silver_customers")

print(f"Version {CANDIDATE_VERSION} row count: {check.count()}")
print(f"Contains customer_id 521 (Day-2-only)? {check.filter(F.col('customer_id') == 521).count() > 0}")

check.filter(F.col("customer_id").isin([1,2,3,4,5])).select("customer_id", "city").show()

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 13, Finished, Available, Finished, False)

Version 2 row count: 500
Contains customer_id 521 (Day-2-only)? False
+-----------+-------------+
|customer_id|         city|
+-----------+-------------+
|          3|    Jasonview|
|          1|South Bridget|
|          5| Carlsonmouth|
|          2|  Herrerafurt|
|          4|  Fostermouth|
+-----------+-------------+



In [12]:
CORRECT_DAY1_VERSION = 2

silver_customers_day1 = spark.read.format("delta").option("versionAsOf", CORRECT_DAY1_VERSION).table("silver_customers")
print(f"Recovered Day 1 snapshot: {silver_customers_day1.count()} rows")

dim_customers_baseline_corrected = (
    silver_customers_day1
    .withColumn("customer_key", F.monotonically_increasing_id().cast(IntegerType()))
    .withColumn("effective_start_date", F.lit(date(2026, 8, 26)).cast(DateType()))
    .withColumn("effective_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .select(
        "customer_key", "customer_id", "first_name", "last_name", "email", "phone",
        "city", "country", "effective_start_date", "effective_end_date", "is_current",
    )
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

dim_customers_baseline_corrected.write.format("delta").mode("overwrite").saveAsTable("dim_customers")
print(f"dim_customers corrected baseline: {dim_customers_baseline_corrected.count()} rows")

print("\n--- Confirming customer_id 1-5 show ORIGINAL Day 1 cities ---")
spark.table("dim_customers").filter(F.col("customer_id").isin([1,2,3,4,5])).select(
    "customer_id", "city", "is_current"
).show()

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 14, Finished, Available, Finished, False)

Recovered Day 1 snapshot: 500 rows
dim_customers corrected baseline: 500 rows

--- Confirming customer_id 1-5 show ORIGINAL Day 1 cities ---
+-----------+-------------+----------+
|customer_id|         city|is_current|
+-----------+-------------+----------+
|          3|    Jasonview|      true|
|          1|South Bridget|      true|
|          5| Carlsonmouth|      true|
|          2|  Herrerafurt|      true|
|          4|  Fostermouth|      true|
+-----------+-------------+----------+



In [13]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window

silver_customers_day2 = spark.table("silver_customers")
dim_customers_current = spark.table("dim_customers").filter(F.col("is_current") == True)

TODAY = date(2026, 8, 27)  # Day 2 processing date

# Step 1: Detect change type per customer (NEW / CHANGED / UNCHANGED)
change_detection = silver_customers_day2.alias("src").join(
    dim_customers_current.alias("cur"),
    on="customer_id",
    how="left"
).select(
    F.col("src.customer_id"),
    F.col("src.first_name"), F.col("src.last_name"), F.col("src.email"),
    F.col("src.phone"), F.col("src.city"), F.col("src.country"),
    F.col("cur.customer_key").alias("existing_key"),
    F.col("cur.city").alias("current_city"),
    F.when(F.col("cur.customer_key").isNull(), F.lit("NEW"))
     .when(F.col("src.city") != F.col("cur.city"), F.lit("CHANGED"))
     .otherwise(F.lit("UNCHANGED"))
     .alias("change_type")
)

print("Change detection summary:")
change_detection.groupBy("change_type").count().show()

# Step 2: Expire old versions for CHANGED customers
dim_customers_table = DeltaTable.forName(spark, "dim_customers")

changed_ids = [
    row["customer_id"] for row in
    change_detection.filter(F.col("change_type") == "CHANGED").select("customer_id").collect()
]

if changed_ids:
    dim_customers_table.update(
        condition=(F.col("is_current") == True) & (F.col("customer_id").isin(changed_ids)),
        set={
            "is_current": F.lit(False),
            "effective_end_date": F.lit(TODAY),
        }
    )
    print(f"Expired {len(changed_ids)} old customer versions.")
else:
    print("No changed customers detected — nothing expired.")

# Step 3: Insert new current rows for CHANGED and NEW customers
existing_max_key = spark.table("dim_customers").agg(F.max("customer_key")).collect()[0][0]

new_versions = (
    change_detection.filter(F.col("change_type").isin(["CHANGED", "NEW"]))
    .select("customer_id", "first_name", "last_name", "email", "phone", "city", "country")
    .withColumn("row_num", F.row_number().over(Window.orderBy("customer_id")))
    .withColumn("customer_key", (F.lit(existing_max_key + 1) + F.col("row_num") - 1).cast(IntegerType()))
    .withColumn("effective_start_date", F.lit(TODAY).cast(DateType()))
    .withColumn("effective_end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
    .drop("row_num")
)

new_versions.write.format("delta").mode("append").saveAsTable("dim_customers")
print(f"Inserted {new_versions.count()} new current versions (changed + brand-new customers).")

# Step 4: Verify SCD2 history for a known changed customer
print("\n--- Verifying SCD2 history for customer_id = 1 ---")
spark.table("dim_customers").filter(F.col("customer_id") == 1).select(
    "customer_key", "customer_id", "city", "is_current", "effective_start_date", "effective_end_date"
).orderBy("effective_start_date").show(truncate=False)

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 15, Finished, Available, Finished, False)

Change detection summary:
+-----------+-----+
|change_type|count|
+-----------+-----+
|    CHANGED|   15|
|        NEW|   10|
|  UNCHANGED|  485|
+-----------+-----+

Expired 15 old customer versions.
Inserted 0 new current versions (changed + brand-new customers).

--- Verifying SCD2 history for customer_id = 1 ---
+------------+-----------+-------------+----------+--------------------+------------------+
|customer_key|customer_id|city         |is_current|effective_start_date|effective_end_date|
+------------+-----------+-------------+----------+--------------------+------------------+
|121         |1          |South Bridget|false     |2026-08-26          |2026-08-27        |
|500         |1          |Jamesburgh   |true      |2026-08-27          |NULL              |
+------------+-----------+-------------+----------+--------------------+------------------+



In [14]:
print(f"Total dim_customers rows: {spark.table('dim_customers').count()}")
print(f"Current rows (is_current=True): {spark.table('dim_customers').filter(F.col('is_current')==True).count()}")
print(f"Expired rows (is_current=False): {spark.table('dim_customers').filter(F.col('is_current')==False).count()}")

# Should be exactly 25: 15 changed customers + 10 new customers
spark.table("dim_customers").filter(F.col("effective_start_date") == date(2026,8,27)).select(
    "customer_id", "city", "is_current"
).orderBy("customer_id").show(30)

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 16, Finished, Available, Finished, False)

Total dim_customers rows: 525
Current rows (is_current=True): 510
Expired rows (is_current=False): 15
+-----------+-------------------+----------+
|customer_id|               city|is_current|
+-----------+-------------------+----------+
|          1|         Jamesburgh|      true|
|          2|          East Dean|      true|
|          3|  West Jessicahaven|      true|
|          4|   New Tracychester|      true|
|          5|        Port Rachel|      true|
|          6|        Lynnborough|      true|
|          7|        South Donna|      true|
|          8|   Christopherhaven|      true|
|          9|    South Leslieton|      true|
|         10|      Angelachester|      true|
|         11|      West Veronica|      true|
|         12| South Aprilborough|      true|
|         13|         Davilastad|      true|
|         14|     East Stephanie|      true|
|         15|        Lake Latoya|      true|
|        521|     South Jonathan|      true|
|        522|           New Luis|      true

**dim_customers — Late-Arriving Dimension Handling (Inferred Members)**

Orders 2001–2003 reference customer_ids 9001–9003, which don't exist in
`dim_customers` — a realistic scenario where the order arrived from a
decoupled source before the corresponding customer record synced (see
ADR-005).

Rather than let these facts fail to join or drop them, we insert
**inferred member** rows: placeholder dimension rows with the known
`customer_id`, all descriptive attributes set to `"Unknown"` / null, and
a flag marking them as inferred. This lets `fact_orders` join successfully
*now*. When the real customer record eventually arrives in a future
Silver run, the standard SCD2 merge logic (Cell 7) will detect it as a
`CHANGED` customer (since "Unknown" ≠ real attributes) and correctly
version it — the inferred row becomes historical, the real data becomes
current. No special-case code is needed for that correction; it falls out
naturally from the existing SCD2 logic.

In [16]:
orders_current = spark.table("silver_orders")
dim_customers_all_ids = spark.table("dim_customers").select("customer_id").distinct()

late_arriving_ids = (
    orders_current.select("customer_id").distinct()
    .join(dim_customers_all_ids, on="customer_id", how="left_anti")
)

late_ids_list = [row["customer_id"] for row in late_arriving_ids.collect()]
print(f"Late-arriving customer_ids found in orders but not in dim_customers: {late_ids_list}")

if late_ids_list:
    existing_max_key = spark.table("dim_customers").agg(F.max("customer_key")).collect()[0][0]

    inferred_members = spark.createDataFrame(
        [(cid,) for cid in late_ids_list], ["customer_id"]
    ).withColumn("customer_id", F.col("customer_id").cast(IntegerType())) \
     .withColumn("row_num", F.row_number().over(Window.orderBy("customer_id"))) \
     .withColumn("customer_key", (F.lit(existing_max_key + 1) + F.col("row_num") - 1).cast(IntegerType())) \
     .withColumn("first_name", F.lit("Unknown")) \
     .withColumn("last_name", F.lit("Unknown")) \
     .withColumn("email", F.lit(None).cast("string")) \
     .withColumn("phone", F.lit(None).cast("string")) \
     .withColumn("city", F.lit("Unknown")) \
     .withColumn("country", F.lit("Unknown")) \
     .withColumn("effective_start_date", F.lit(TODAY).cast(DateType())) \
     .withColumn("effective_end_date", F.lit(None).cast(DateType())) \
     .withColumn("is_current", F.lit(True)) \
     .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType())) \
     .drop("row_num")

    inferred_count = inferred_members.count()
    inferred_members.write.format("delta").mode("append").saveAsTable("dim_customers")
    print(f"Inserted {inferred_count} inferred member row(s) for late-arriving customers.")
else:
    print("No late-arriving customers found — nothing to infer.")

print("\n--- Inferred member rows ---")
spark.table("dim_customers").filter(F.col("customer_id").isin([9001, 9002, 9003])).select(
    "customer_key", "customer_id", "first_name", "city", "is_current", "effective_start_date"
).show()

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 19, Finished, Available, Finished, False)

Late-arriving customer_ids found in orders but not in dim_customers: [9001, 9002, 9003]
Inserted 3 inferred member row(s) for late-arriving customers.

--- Inferred member rows ---
+------------+-----------+----------+-------+----------+--------------------+
|customer_key|customer_id|first_name|   city|is_current|effective_start_date|
+------------+-----------+----------+-------+----------+--------------------+
|         525|       9001|   Unknown|Unknown|      true|          2026-08-27|
|         526|       9002|   Unknown|Unknown|      true|          2026-08-27|
|         527|       9003|   Unknown|Unknown|      true|          2026-08-27|
+------------+-----------+----------+-------+----------+--------------------+



In [17]:
print(f"dim_customers total rows: {spark.table('dim_customers').count()}")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 20, Finished, Available, Finished, False)

dim_customers total rows: 528


**fact_orders**

Grain: one row per order line item (order_item).

Joins:
- `dim_customers` — via `customer_id`, filtered to `is_current = true`
  (acceptable simplification here since we're building the fact after
  the SCD2 merge; a stricter point-in-time join using
  `effective_start_date`/`effective_end_date` against `order_date` is
  discussed below as the production-correct approach).
- `dim_products` — via `product_id`
- No FX join yet — currency conversion is out of scope for this fact
  table's initial grain (all seeded prices are USD-equivalent).

This table finally proves the late-arriving dimension fix worked: orders
2001-2003 should appear here, correctly joined to their `Unknown`
inferred-member customer rows rather than being dropped.

In [18]:
silver_orders = spark.table("silver_orders")
silver_order_items = spark.table("silver_order_items")
dim_customers_current = spark.table("dim_customers").filter(F.col("is_current") == True)
dim_products_lookup = spark.table("dim_products")

fact_orders = (
    silver_order_items.alias("oi")
    .join(silver_orders.alias("o"), "order_id", "inner")
    .join(dim_customers_current.alias("dc"), "customer_id", "left")
    .join(dim_products_lookup.alias("dp"), "product_id", "left")
    .select(
        F.col("oi.order_item_id"),
        F.col("o.order_id"),
        F.col("dc.customer_key"),
        F.col("dp.product_key"),
        F.col("o.order_date"),
        F.col("o.order_status"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        (F.col("oi.quantity") * F.col("oi.unit_price")).alias("line_total"),
    )
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

fact_orders.write.format("delta").mode("overwrite").saveAsTable("fact_orders")
print(f"fact_orders: {fact_orders.count()} rows")

# Verify the late-arriving orders successfully joined via inferred members
print("\n--- Verifying late-arriving orders joined correctly ---")
fact_orders.filter(F.col("order_id").isin([2001, 2002, 2003])).show()

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 21, Finished, Available, Finished, False)

fact_orders: 5896 rows

--- Verifying late-arriving orders joined correctly ---
+-------------+--------+------------+-----------+-------------------+------------+--------+----------+----------+--------------------+
|order_item_id|order_id|customer_key|product_key|         order_date|order_status|quantity|unit_price|line_total|  _gold_processed_at|
+-------------+--------+------------+-----------+-------------------+------------+--------+----------+----------+--------------------+
|         6025|    2001|         525|         14|2026-08-27 08:14:31|     pending|       3|    482.64|   1447.92|2026-08-28 08:02:...|
|         6026|    2002|         526|         88|2026-08-27 08:14:31|     pending|       1|    145.48|    145.48|2026-08-28 08:02:...|
|         6027|    2003|         527|         85|2026-08-27 08:14:31|     pending|       2|    396.18|    792.36|2026-08-28 08:02:...|
+-------------+--------+------------+-----------+-------------------+------------+--------+----------+--------

**fact_reviews**

Grain: one row per review. Joins to `dim_products` to attribute each
review to a product. Reviews with no valid product reference were already
quarantined in Silver (Phase 6), so this join should have zero nulls on
`product_key` — worth verifying explicitly.

In [19]:
silver_reviews = spark.table("silver_reviews")
dim_products_lookup = spark.table("dim_products")

fact_reviews = (
    silver_reviews.alias("r")
    .join(dim_products_lookup.alias("dp"), "product_id", "left")
    .select(
        F.col("r.review_id"),
        F.col("dp.product_key"),
        F.col("r.reviewer_name"),
        F.col("r.verified_purchase"),
        F.col("r.reviewer_city"),
        F.col("r.reviewer_country"),
        F.col("r.rating"),
        F.col("r.has_missing_rating"),
        F.col("r.review_text"),
        F.col("r.review_date"),
        F.col("r.helpful_votes"),
        F.col("r.tags"),
    )
    .withColumn("_gold_processed_at", F.lit(GOLD_RUN_TS).cast(TimestampType()))
)

fact_reviews.write.format("delta").mode("overwrite").saveAsTable("fact_reviews")
print(f"fact_reviews: {fact_reviews.count()} rows")

null_product_key_count = fact_reviews.filter(F.col("product_key").isNull()).count()
print(f"Reviews with null product_key (should be 0, since bad refs were quarantined in Silver): {null_product_key_count}")

StatementMeta(, ec6c724b-d673-49bb-90ff-9e3c21a9df0c, 22, Finished, Available, Finished, False)

fact_reviews: 394 rows
Reviews with null product_key (should be 0, since bad refs were quarantined in Silver): 0
